In [4]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

sys.path.append("../src")

from master_thesis.stage2 import (
    load_stage2_config,
    resolve_stage2_paths,
    load_stage1_preprocessor,
    get_feature_cols,
    build_stage2_model_from_config,
    load_stage1_model_weights,
)

from master_thesis.episode_sampler import build_department_task_table, filter_valid_departments, make_logistics_meta_test_split
from master_thesis.metrics import evaluate_on_gold_binary
from master_thesis.plotting import plot_roc_curve, plot_precision_recall_curve

CONFIG_PATH = "../experiments/stage2_config.yaml"
# Ensure we are using the new fixed 226 column CSV!
DATA_PATH = "../Data/processed/contract_with_features_labeled_stage2_test.csv"

# Load in the Configurations
config = load_stage2_config(CONFIG_PATH)
paths = resolve_stage2_paths(config)

df = pd.read_csv(DATA_PATH, low_memory=False)
preprocessor = load_stage1_preprocessor(paths.stage1_preprocessor_path)
feature_cols = get_feature_cols(config, preprocessor=preprocessor)

print(f"Dataset shape loaded globally: {df.shape}")
dept_col = config["data"]["department_col"]
print(f"Number of distinct departments available: {df[dept_col].nunique()}")
print(f"Meta-Testing specifically targeted to: {config['task_config']['target_department']}")

# Build the foundational Task Dataframe
task_df = build_department_task_table(
    features_df=df,
    feature_cols=feature_cols,
    contract_id_col=config["data"]["group_col"],
    department_col=config["data"]["department_col"],
    target_col=config["data"]["target_col"],
    observation_year_col=config["data"]["observation_year_col"],
    drop_missing_features=False,
)

# Mathematically partition Logistics into 10-20 distinct random Evaluation Splits
target_episodes = make_logistics_meta_test_split(
    task_df=task_df,
    feature_cols=feature_cols,
    target_department=config["task_config"]["target_department"],
    department_col=config["data"]["department_col"],
    target_col=config["data"]["target_col"],
    contract_id_col=config["data"]["group_col"],
    n_support_pos=config["support_config"]["n_support_pos"],
    n_support_neg=config["support_config"]["n_support_neg"],
    n_repeats=config["meta_config"]["n_repeats"],
)


Dataset shape loaded globally: (9201, 230)
Number of distinct departments available: 16
Meta-Testing specifically targeted to: Logistics
Stage 2 task table created
Rows: 180
Unique contracts: 40
Departments: 8
Positive labels: 100
Negative labels: 80


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
target_col = getattr(config, "gold_col", "gold_y")

def prepare_episode_tensors(
    support_df: pd.DataFrame,
    query_df: pd.DataFrame,
    feature_cols: list[str],
    preprocessor,
    target_col: str,
    device: torch.device,
):
    X_support_raw = support_df[feature_cols].copy()
    X_query_raw = query_df[feature_cols].copy()

    # Pass strings through the global Stage 1 preprocessor before converting to Pytorch Tensors
    X_support = preprocessor.transform(X_support_raw)
    X_query = preprocessor.transform(X_query_raw)

    if hasattr(X_support, "toarray"):
        X_support = X_support.toarray()
    if hasattr(X_query, "toarray"):
        X_query = X_query.toarray()

    y_support = support_df[target_col].to_numpy(dtype=np.float32)
    y_query = query_df[target_col].to_numpy(dtype=np.float32)

    X_support_t = torch.tensor(X_support, dtype=torch.float32, device=device)
    X_query_t = torch.tensor(X_query, dtype=torch.float32, device=device)
    y_support_t = torch.tensor(y_support, dtype=torch.float32, device=device).view(-1, 1)
    y_query_t = torch.tensor(y_query, dtype=torch.float32, device=device).view(-1, 1)

    return X_support_t, y_support_t, X_query_t, y_query_t, X_support.shape[1], y_query

def predict_proba(model: nn.Module, X: torch.Tensor) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
    return probs

def extract_inner_lr(config, default=1e-2):
    for attr in ["inner_lr", "adapt_lr", "support_lr", "fast_lr"]:
        if hasattr(config, attr):
            return getattr(config, attr)
    return default

def extract_inner_steps(config, default=5):
    for attr in ["inner_steps", "adapt_steps", "support_steps", "n_inner_steps"]:
        if hasattr(config, attr):
            return getattr(config, attr)
    return default

def clone_model_from_stage1(input_dim: int):
    model = build_stage2_model_from_config(input_dim, config).to(device)
    load_stage1_model_weights(model, paths.stage1_model_path)
    return model

def fomaml_adapt(
    model: nn.Module,
    X_support: torch.Tensor,
    y_support: torch.Tensor,
    inner_lr: float,
    inner_steps: int,
):
    adapted = copy.deepcopy(model)
    adapted.train()
    optimizer = torch.optim.SGD(adapted.parameters(), lr=inner_lr)
    criterion = nn.BCEWithLogitsLoss()
    history = []

    for step in range(inner_steps):
        optimizer.zero_grad()
        logits = adapted(X_support)
        loss = criterion(logits, y_support)
        loss.backward()
        optimizer.step()
        history.append({"step": step + 1, "support_loss": float(loss.item())})

    return adapted, pd.DataFrame(history)


In [7]:
all_episode_metrics = []

print(f"\\n--- EVALUATION BOOTING UP: Testing {len(target_episodes)} Splits ---")

for ep_idx, episode in enumerate(target_episodes):
    print(f"Adapting Model to Logistics Support Split #{ep_idx + 1}...")
    
    # Ready the Data
    support_df, query_df = episode["support_df"], episode["query_df"]
    X_support_t, y_support_t, X_query_t, y_query_t, input_dim, y_query_np = prepare_episode_tensors(
        support_df=support_df, query_df=query_df, feature_cols=feature_cols, 
        preprocessor=preprocessor, target_col=target_col, device=device,
    )
    
    # ---------------------------------------------------------
    # ZERO-SHOT BASELINE
    # ---------------------------------------------------------
    zero_shot_model = clone_model_from_stage1(input_dim)
    zero_shot_metrics = evaluate_on_gold_binary(
        y_query_np, predict_proba(zero_shot_model, X_query_t), "Zero-Shot"
    )
    zero_shot_metrics["episode_idx"] = ep_idx
    all_episode_metrics.append(zero_shot_metrics)

    # ---------------------------------------------------------
    # STANDARD FINE-TUNING
    # ---------------------------------------------------------
    ft_model = clone_model_from_stage1(input_dim)
    optimizer = torch.optim.Adam(
        ft_model.parameters(), lr=extract_inner_lr(config, 1e-3),
        weight_decay=getattr(config, "weight_decay", 1e-4)
    )
    criterion = nn.BCEWithLogitsLoss()
    
    ft_model.train()
    for step in range(extract_inner_steps(config, 10)):
        optimizer.zero_grad()
        loss = criterion(ft_model(X_support_t), y_support_t)
        loss.backward()
        optimizer.step()

    ft_metrics = evaluate_on_gold_binary(
        y_query_np, predict_proba(ft_model, X_query_t), "Fine-Tuning"
    )
    ft_metrics["episode_idx"] = ep_idx
    all_episode_metrics.append(ft_metrics)

    # ---------------------------------------------------------
    # FOMAML ADAPTATION
    # ---------------------------------------------------------
    fomaml_adapted, _ = fomaml_adapt(
        model=clone_model_from_stage1(input_dim),
        X_support=X_support_t, y_support=y_support_t,
        inner_lr=extract_inner_lr(config, 1e-2),
        inner_steps=extract_inner_steps(config, 5),
    )
    
    fomaml_metrics = evaluate_on_gold_binary(
        y_query_np, predict_proba(fomaml_adapted, X_query_t), "FOMAML"
    )
    fomaml_metrics["episode_idx"] = ep_idx
    all_episode_metrics.append(fomaml_metrics)

# ---------------------------------------------------------
# VARIANCE AGGREGATION
# ---------------------------------------------------------
# FIX: Use pd.concat to neatly stack the 60 DataFrames together!
metrics_df = pd.concat(all_episode_metrics, ignore_index=True)
key_metrics = ["gold_auroc", "gold_ap", "gold_brier", "gold_f1"]

# Group by model type, and mathematically aggregate the Mean and Std Deviation
aggregated_metrics = metrics_df.groupby("model")[key_metrics].agg(["mean", "std"])

print("\\n-----------------------------------------------------------")
print("                 FINAL AGGREGATED METRICS")
print("-----------------------------------------------------------")
display(aggregated_metrics)



\n--- EVALUATION BOOTING UP: Testing 20 Splits ---
Adapting Model to Logistics Support Split #1...
Adapting Model to Logistics Support Split #2...
Adapting Model to Logistics Support Split #3...
Adapting Model to Logistics Support Split #4...
Adapting Model to Logistics Support Split #5...


/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Libra

Adapting Model to Logistics Support Split #6...
Adapting Model to Logistics Support Split #7...
Adapting Model to Logistics Support Split #8...
Adapting Model to Logistics Support Split #9...
Adapting Model to Logistics Support Split #10...
Adapting Model to Logistics Support Split #11...


/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Libra

Adapting Model to Logistics Support Split #12...
Adapting Model to Logistics Support Split #13...
Adapting Model to Logistics Support Split #14...
Adapting Model to Logistics Support Split #15...
Adapting Model to Logistics Support Split #16...
Adapting Model to Logistics Support Split #17...


/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Libra

Adapting Model to Logistics Support Split #18...
Adapting Model to Logistics Support Split #19...
Adapting Model to Logistics Support Split #20...
\n-----------------------------------------------------------
                 FINAL AGGREGATED METRICS
-----------------------------------------------------------


/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Library/Caches/pypoetry/virtualenvs/master-thesis-TTOTpqnd-py3.12/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['has_environmental_appendix']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/annita/Libra

gold_auroc             gold_ap           gold_brier            \
                  mean       std      mean       std       mean       std   
model                                                                       
FOMAML        0.624824  0.178414  0.721843  0.193934   0.462374  0.173870   
Fine-Tuning   0.527652  0.096495  0.631791  0.132552   0.441999  0.141352   
Zero-Shot     0.782509  0.183812  0.843957  0.197008   0.207440  0.084353   

              gold_f1            
                 mean       std  
model                            
FOMAML       0.349491  0.390637  
Fine-Tuning  0.365817  0.416066  
Zero-Shot    0.751130  0.138527